# Unidad VII — Práctica en Python (complemento)

**Bases de Datos Orientadas a Grafos**

Cubre lo que **no corre en Cypher/GDS**:

1. **Completado de knowledge graphs** con TransE y PairRE (**PyKEEN**).
2. **Construcción de un grafo desde texto** — NER y extracción de relaciones (**spaCy**) + carga a Neo4j.
3. **GraphRAG** — recuperación de un subgrafo como contexto.

Corre entero en una laptop con **CPU** (no necesita GPU). Para las secciones 2.4 y 3 hay que tener **Neo4j Desktop** corriendo (la misma instancia de la práctica GDS).


## 0. Instalación (ejecutar una vez)

Descarga PyKEEN, spaCy y los conectores, y el modelo de inglés de spaCy. **Después de que termine, reiniciá el kernel** (Kernel → Restart) y seguí desde la sección 1.


In [1]:
#!pip install pykeen spacy neo4j neo4j-graphrag ollama
!python -m spacy download en_core_web_sm

     ---------------------------------------- 0.0/12.8 MB ? eta -:--:--
     ------------ --------------------------- 3.9/12.8 MB 19.5 MB/s eta 0:00:01
     -------------------------- ------------- 8.4/12.8 MB 20.0 MB/s eta 0:00:01
     --------------------------------------  12.6/12.8 MB 20.7 MB/s eta 0:00:01
     --------------------------------------- 12.8/12.8 MB 18.7 MB/s eta 0:00:00
[+] Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')


## 1. Completado de knowledge graphs: TransE y PairRE

Un knowledge graph guarda hechos como **triples** `(cabeza, relación, cola) = (h, r, t)`. Como está incompleto, se aprenden *embeddings* (un vector por entidad y por relación) que **puntúan** qué tan plausible es un triple.

- **TransE**: `f(h,t) = -‖h + r − t‖` (busca `h + r ≈ t`).
- **PairRE**: `f(h,t) = -‖h∘r_H − t∘r_T‖` — modela mejor relaciones 1-N, N-1 y N-N.

Dataset **Nations** (14 países, 55 relaciones), incluido en PyKEEN. Métricas: **MRR** y **Hits@k**.


In [2]:
from pykeen.pipeline import pipeline

res_transe = pipeline(
    dataset='Nations', model='TransE',
    model_kwargs=dict(embedding_dim=64),
    training_kwargs=dict(num_epochs=100, use_tqdm_batch=False),
    random_seed=42,
)
print('TransE  -> MRR:', round(res_transe.get_metric('mrr'), 3),
      '| Hits@10:', round(res_transe.get_metric('hits@10'), 3))

No cuda devices were available. The model runs on CPU
C:\Users\josei\anaconda3\envs\jupyter_classic\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


Training epochs on cpu:   0%|          | 0/100 [00:00<?, ?epoch/s]

Evaluating on cpu:   0%|          | 0.00/201 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.29s seconds


TransE  -> MRR: 0.346 | Hits@10: 0.98


In [3]:
res_pairre = pipeline(
    dataset='Nations', model='PairRE',
    model_kwargs=dict(embedding_dim=64),
    training_kwargs=dict(num_epochs=100, use_tqdm_batch=False),
    random_seed=42,
)
print('PairRE  -> MRR:', round(res_pairre.get_metric('mrr'), 3),
      '| Hits@10:', round(res_pairre.get_metric('hits@10'), 3))
print('\nMRR  TransE vs PairRE:', round(res_transe.get_metric('mrr'),3),
      'vs', round(res_pairre.get_metric('mrr'),3),
      ' (PairRE suele superar a TransE en relaciones N-N)')

INFO:pykeen.datasets.utils:Loading cached preprocessed dataset from file:///C:/Users/josei/.data/pykeen/datasets/nations/cache/47DEQpj8HBSa-_TImW-5JCeuQeRkm5NM
INFO:pykeen.triples.triples_factory:Loading from file:///C:/Users/josei/.data/pykeen/datasets/nations/cache/47DEQpj8HBSa-_TImW-5JCeuQeRkm5NM/training
INFO:pykeen.triples.triples_factory:Loading from file:///C:/Users/josei/.data/pykeen/datasets/nations/cache/47DEQpj8HBSa-_TImW-5JCeuQeRkm5NM/testing
INFO:pykeen.triples.triples_factory:Loading from file:///C:/Users/josei/.data/pykeen/datasets/nations/cache/47DEQpj8HBSa-_TImW-5JCeuQeRkm5NM/validation
INFO:pykeen.pipeline.api:Using device: None
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
INFO:pykeen.nn.representation:Inferred unique=False for Embedding()
C:\Users\josei\anaconda3\envs\jupyter_classic\Lib\site-packages\torch\utils\data\dataloader.py:665: UserWarning: 'pin_memory' argument is set

Training epochs on cpu:   0%|          | 0/100 [00:00<?, ?epoch/s]

Evaluating on cpu:   0%|          | 0.00/201 [00:00<?, ?triple/s]

INFO:pykeen.evaluation.evaluator:Evaluation took 0.20s seconds


PairRE  -> MRR: 0.728 | Hits@10: 0.993

MRR  TransE vs PairRE: 0.346 vs 0.728  (PairRE suele superar a TransE en relaciones N-N)


### Predicción de hechos faltantes

Para un `(h, r, ?)` se rankean todas las colas candidatas: eso es 'completar' el grafo.


In [4]:
from pykeen.predict import predict_target

print('Entidades :', list(res_transe.training.entity_to_id)[:8])
print('Relaciones:', list(res_transe.training.relation_to_id)[:8])

pred = predict_target(model=res_pairre.model, head='brazil',
                      relation='intergovorgs', triples_factory=res_pairre.training)
pred.df.head(10)

Entidades : ['brazil', 'burma', 'china', 'cuba', 'egypt', 'india', 'indonesia', 'israel']
Relaciones: ['accusation', 'aidenemy', 'attackembassy', 'blockpositionindex', 'booktranslations', 'boycottembargo', 'commonbloc0', 'commonbloc1']


,tail_id,score,tail_label
12,12,-0.946535,usa
9,9,-0.969873,netherlands
11,11,-1.039884,uk
5,5,-1.121939,india
10,10,-1.211088,poland
7,7,-1.352279,israel
13,13,-1.511988,ussr
0,0,-1.575636,brazil
4,4,-1.604685,egypt
3,3,-1.649029,cuba


## 2. Construcción de un grafo desde texto

`Texto → NER → extracción de relaciones → carga al grafo`. (La correferencia y el *entity linking* se comentan abajo; en producción se resuelven con modelos dedicados o un LLM.)


### 2.1 NER — reconocimiento de entidades (spaCy)


In [5]:
import spacy
nlp = spacy.load('en_core_web_sm')

texto = ('Tim Berners-Lee founded the World Wide Web Consortium at MIT in 1994. '
         'Tim Berners-Lee directed the organization for many years.')
doc = nlp(texto)

print('Entidades detectadas (candidatas a nodos):')
for ent in doc.ents:
    print(f'  {ent.text:35s} -> {ent.label_}')

Entidades detectadas (candidatas a nodos):
  Tim Berners-Lee                     -> PERSON
  the World Wide Web Consortium       -> EVENT
  MIT                                 -> ORG
  1994                                -> DATE
  Tim Berners-Lee                     -> PERSON
  many years                          -> DATE


### 2.2 Extracción de relaciones (dependencias)

Extracción simple sujeto–verbo–objeto del árbol de dependencias. Cada triple es una arista tipada.


In [6]:
def span(w):
    return doc[w.left_edge.i : w.right_edge.i+1].text

def extraer_triples(doc):
    triples = []
    for tok in doc:
        if tok.pos_ == 'VERB':
            sujetos = [w for w in tok.lefts  if w.dep_ in ('nsubj','nsubjpass')]
            objetos = [w for w in tok.rights if w.dep_ in ('dobj','attr','dative')]
            preps   = [w for w in tok.rights if w.dep_ == 'prep']
            for s in sujetos:
                # objeto directo: usamos el verbo TAL CUAL (no el lema)
                for o in objetos:
                    triples.append((span(s), tok.text, span(o)))
                # objetos de preposicion: 'founded_at MIT', 'founded_in 1994'
                for p in preps:
                    for po in [w for w in p.rights if w.dep_ == 'pobj']:
                        triples.append((span(s), f'{tok.text}_{p.text}', span(po)))
    return triples

triples = extraer_triples(doc)
for h, r, t in triples:
    print(f'({h})  -[{r}]->  ({t})')

(Tim Berners-Lee)  -[founded]->  (the World Wide Web Consortium)
(Tim Berners-Lee)  -[founded_at]->  (MIT)
(Tim Berners-Lee)  -[founded_in]->  (1994)
(Tim Berners-Lee)  -[directed]->  (the organization)
(Tim Berners-Lee)  -[directed_for]->  (many years)


> **Correferencia y entity linking.** spaCy base no resuelve correferencia (por eso en el texto repetimos 'Tim Berners-Lee' en vez de 'He'); en un caso real se usa `fastcoref`/`coreferee` o un LLM. El *entity linking* mapea cada mención a una entidad canónica (p. ej. 'MIT' → Wikidata `Q49108`) para unificar nodos entre documentos. Ambas suelen resolverse hoy con un LLM que devuelve entidades y triples en JSON.


### 2.3 Conexión a Neo4j (instancia local)

Poné la contraseña de tu Neo4j Desktop. Tiene que estar **corriendo** la instancia.


In [8]:
from neo4j import GraphDatabase

NEO4J_URI      = 'bolt://localhost:7687'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'Zorril44' #'tu_password'   # <-- poné tu contraseña de Neo4j Desktop

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()
print('Conectado a Neo4j.')

Conectado a Neo4j.


### 2.4 Cargar los triples en Neo4j


In [9]:
def cargar(tx, h, r, t):
    tx.run('MERGE (a:Entidad {nombre:$h}) '
           'MERGE (b:Entidad {nombre:$t}) '
           'MERGE (a)-[rel:REL {tipo:$r}]->(b)', h=h, r=r, t=t)

with driver.session() as s:
    s.run('MATCH (e:Entidad) DETACH DELETE e')   # limpiar una corrida anterior de la demo
    for h, r, t in triples:
        s.execute_write(cargar, h, r, t)
print(f'{len(triples)} triples cargados como nodos :Entidad y relaciones :REL.')

5 triples cargados como nodos :Entidad y relaciones :REL.


## 3. GraphRAG — recuperar un subgrafo como contexto

El **RAG vectorial** recupera pasajes de texto sueltos. **GraphRAG** recupera un **subgrafo** (entidades relacionadas y caminos): contexto conectado, razonamiento multi-hop y respuestas trazables.

Acá hacemos la parte que distingue a GraphRAG —la **recuperación**— con tu Neo4j: dada una entidad, traemos su subgrafo y armamos el contexto que se le pasaría al LLM.


In [10]:
# El grafo del ejemplo es diminuto: recuperamos TODOS los hechos como contexto.
# En un grafo grande se acota a un subgrafo alrededor de la entidad de la pregunta,
# p. ej.:  MATCH (e:Entidad {nombre:$entidad})-[r:REL*1..2]-(v) ...
CYPHER = '''
MATCH (a:Entidad)-[r:REL]->(b:Entidad)
RETURN a.nombre AS origen, r.tipo AS relacion, b.nombre AS destino'''

def subgrafo_a_texto(registros):
    return '\n'.join(f'- {r["origen"]} {r["relacion"]} {r["destino"]}' for r in registros)

with driver.session() as s:
    regs = list(s.run(CYPHER))

contexto = subgrafo_a_texto(regs)
print('Hechos recuperados (contexto para el LLM):\n')
print(contexto if contexto else '(grafo vacío: corré antes la celda 2.4 de carga)')

Hechos recuperados (contexto para el LLM):

- Tim Berners-Lee founded the World Wide Web Consortium
- Tim Berners-Lee founded_at MIT
- Tim Berners-Lee founded_in 1994
- Tim Berners-Lee directed the organization
- Tim Berners-Lee directed_for many years


### 3.2 Generar la respuesta con un LLM local (Ollama)

La recuperación de arriba es el corazón de GraphRAG; la **generación** necesita un LLM. Para no depender de ninguna API key, usamos **Ollama** (modelos locales, gratis). La librería `ollama` ya se instaló en la celda 0.

**Lo único que no es `pip`:** instalar la **app Ollama** desde ollama.com y dejarla corriendo (es un programa aparte, el motor que ejecuta los modelos). La primera celda descarga el modelo y la segunda genera la respuesta.


In [11]:
import ollama
ollama.pull('llama3.2:3b')   # descarga el modelo si falta (rapido si ya esta)
print('Modelo listo.')

Modelo listo.


In [12]:
import ollama

pregunta = '¿Quién fundó el W3C y dónde?'
prompt = f"""Respondé la pregunta usando ÚNICAMENTE estos hechos del grafo:
{contexto}

Pregunta: {pregunta}
Respuesta:"""

resp = ollama.chat(model='llama3.2:3b', messages=[{'role': 'user', 'content': prompt}])
print(resp['message']['content'])

Tim Berners-Lee fundó el W3C en la MIT en 1994.


> **Alternativa con API key.** Si preferís un modelo hospedado, `neo4j-graphrag` trae `OpenAILLM` (requiere `OPENAI_API_KEY`) y otros proveedores (Groq, Gemini y Mistral tienen *free tier*). La lógica es la misma: contexto + pregunta → LLM.


### Limpieza (opcional)


In [13]:
# Borrar los nodos :Entidad creados por esta demo (no afecta al grafo :User de la práctica GDS)
with driver.session() as s:
    s.run('MATCH (e:Entidad) DETACH DELETE e')
driver.close()
print('Listo: nodos de la demo eliminados y conexión cerrada.')

Listo: nodos de la demo eliminados y conexión cerrada.


## Cierre

- **PyKEEN**: entrena y evalúa KG embeddings (TransE, PairRE) y predice hechos faltantes.
- **Texto → grafo** (spaCy): extrae entidades y relaciones y las materializa en Neo4j.
- **GraphRAG**: recupera un subgrafo del grafo como contexto conectado para un LLM.

Referencias: Bratanič, *Graph Algorithms for Data Science* (caps. 11–12); PyKEEN (pykeen.readthedocs.io); documentación de neo4j-graphrag; Edge et al., *From Local to Global: A Graph RAG Approach* (2024).
